# 1. Anyons in the wavefunction

What *is* an anyon, as a property of the many-qubit wavefunction? This notebook answers it on a $3 \times 3$ torus
(18 qubits), where we can also write the whole state vector down and check every claim exactly
(`toric.ExactToricCode`).

In [1]:
import matplotlib.pyplot as plt
import numpy as np

import toric

BLUE, ORANGE = "#2a78d6", "#eb6834"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.figsize": (5.5, 3.4)})

L = 3
ex = toric.ExactToricCode(L)
t = ex.torus
print(f"{t.n_edges} qubits, state vectors of length {2**t.n_edges}")

18 qubits, state vectors of length 262144


## The ground state is a soup of closed loops

Write each qubit in the $Z$ basis, $|0\rangle$ or $|1\rangle$. $B_p = +1$ means an even number of $|1\rangle$
edges around every square. Draw each $|1\rangle$ edge as a short segment *across* the edge (on the dual lattice):
then these segments always join into **closed loops**. $A_v = +1$ means that flipping the 4 edges at a vertex,
which adds a tiny loop, does not change the amplitude. So a ground state is an equal superposition of all
closed-loop pictures.

In [2]:
psi0 = ex.psi0
support = np.flatnonzero(np.abs(psi0) > 1e-12)
print(f"{len(support)} pictures, all with |amplitude|^2 = {abs(psi0[support[0]])**2:.6f} = 1/{len(support)}")
print("energy:", ex.energy(psi0), " (= -2 L^2 =", -2 * L * L, ")")
print("ground-space dimension: the 4 states", "are orthonormal:", np.allclose(ex.basis.conj() @ ex.basis.T, np.eye(4)))

256 pictures, all with |amplitude|^2 = 0.003906 = 1/256
energy: -18.0  (= -2 L^2 = -18 )
ground-space dimension: the 4 states are orthonormal: True


The other three ground states are the same loop soup with one or two extra loops that wrap around the torus:
there are $2^{L^2 - 1} = 256$ pictures per ground state here, and 4 ground states.

## An m anyon is the end of a loop

$X$ on an edge flips it between $|0\rangle$ and $|1\rangle$ in **every** picture. So every picture now has an open
string, ending at the two squares beside that edge. At those squares the number of $|1\rangle$ edges is odd:
$B_p = -1$. That is an **m anyon**, and you can see it in each single picture.

In [3]:
s = toric.new_state(L)
s.apply("X", t.h(1, 1))
rng = np.random.default_rng(1)
for _ in range(3):
    base, config, z, amp = s.sample_picture([1, 0, 0, 0], rng)
    counts = [int(config[t.plaquette(i, j)].sum()) for j in range(L) for i in range(L)]
    print("|1> edges around each square:", counts, " -> odd at", [k for k, n in enumerate(counts) if n % 2])
print("m anyons (plaquettes j L + i):", [j * L + i for i, j in s.anyons()[1]])

|1> edges around each square: [2, 1, 2, 2, 3, 2, 4, 4, 2]  -> odd at [1, 4]
|1> edges around each square: [2, 3, 2, 0, 1, 0, 2, 2, 2]  -> odd at [1, 4]
|1> edges around each square: [2, 3, 2, 2, 3, 2, 0, 2, 2]  -> odd at [1, 4]
m anyons (plaquettes j L + i): [1, 4]


## An e anyon is a sign twist between pictures

$Z$ on an edge does not change any picture; it multiplies the amplitude by $-1$ for every picture in which that edge
is $|1\rangle$. Now compare two pictures that differ only by the small loop around one vertex (that is what $A_v$
does). Normally they have the same amplitude. At the two vertices at the ends of the $Z$ they have **opposite
signs**: $A_v = -1$. No single picture shows an e anyon; it lives in the relative signs.

In [4]:
s = toric.new_state(L)
s.apply("Z", t.h(0, 0))                       # e anyons at vertices (0, 0) and (1, 0)
psi = ex.apply("Z", t.h(0, 0), ex.psi0)
base, config, z, amp = s.sample_picture([1, 0, 0, 0], rng)
for vertex in [(0, 0), (1, 0), (2, 2)]:
    flipped = config.copy()
    flipped[t.star(*vertex)] ^= 1
    ratio = ex.amplitude(flipped, psi) / ex.amplitude(config, psi)
    print(f"vertex {vertex}: amplitude ratio after flipping its star = {ratio.real:+.0f}")

vertex (0, 0): amplitude ratio after flipping its star = -1
vertex (1, 0): amplitude ratio after flipping its star = -1
vertex (2, 2): amplitude ratio after flipping its star = +1


## The string is invisible

Two strings with the same end points differ by a closed loop. A small closed $Z$ loop is a plaquette $B_p$, which is
$+1$ on the ground state, so it changes nothing. Only where the anyons are matters, plus how the strings wrap the
torus (notebook 2).

In [5]:
short = ex.apply("Z", t.h(0, 0), ex.psi0)                          # Z on one edge
detour = ex.psi0
for e in [t.v(0, 0), t.h(0, 1), t.v(1, 0)]:                         # the same two end points, the long way round
    detour = ex.apply("Z", e, detour)
print("overlap of the two states:", np.round(np.vdot(short, detour), 12))

overlap of the two states: (1+0j)


## Braiding: an e taken around an m gives $-1$

$Z$ and $X$ on the same qubit anticommute, $ZX = -XZ$. Make an m pair, then take an e around one m: its closed $Z$
loop crosses the m's $X$ string exactly once. Annihilate everything and you are back in the start state, but with
a minus sign. This $-1$ is the **mutual statistics** of e and m: neither is a boson or a fermion relative to the
other.

In [6]:
seq = [("X", t.h(1, 1))] + [("Z", e) for e in t.plaquette(1, 1)] + [("X", t.h(1, 1))]
psi = ex.psi0
s = toric.new_state(L)
for gate, e in seq:
    psi = ex.apply(gate, e, psi)
    s.apply(gate, e)
print("exact  <start|end> =", np.round(np.vdot(ex.psi0, psi), 12))
print("toric  <start|end> =", s.logical([1, 0, 0, 0])[0])

exact  <start|end> = (-1+0j)
toric  <start|end> = (-1+0j)


The same holds on any torus: `toric` tracks every state as $i^k X^a Z^b|\text{start}\rangle$ (two bit strings and a
phase), and `tests/test_code.py` checks energies, stabilizers, amplitudes, loops and logical states against these
exact state vectors.

## Exercises

1. Apply $Y$ on one edge. Which stabilizers flip? Check with `ex.stabilizers(psi)`.
2. Take an e around an m **twice**. What is the phase?
3. Take an e around a square that contains *no* m. Show the phase is $+1$, and explain it with the loop soup.
4. Count the pictures in one ground state for $L = 2$ and $L = 3$ and compare with $2^{L^2 - 1}$.